# 18. Incremental LS-RQAOA и MK-стратегии

Эпоха, банк, age, четыре score и частичное обновление.

In [1]:
import json
from itertools import combinations
from pathlib import Path
import numpy as np
from QA_max_cut_lib import WeightedGraph
from QA_max_cut_lib.generators.incremental import survivor_neighbor_cover
from QA_max_cut_lib.solvers.incremental_ls import IncrementalLSRQAOACPU
ROOT = Path.cwd()
if not (ROOT / 'configs').exists(): ROOT = ROOT.parent
labels = [-7,10,30,80,101,205]
pairs = list(combinations(labels, 2))
graph = WeightedGraph([(u,v,(2*i+1)/8) for i,(u,v) in enumerate(pairs)], nodes=labels)

## Одна эпоха: mean, zero и missing

In [2]:
config = json.loads((ROOT/'configs/solvers/incremental_ls_current.json').read_text())
config['output'] = {'report': 0}
config['limits']['max_reductions'] = 1
solver = IncrementalLSRQAOACPU(config); solver.solve(graph, num_treads=1)
e = solver.result['epoch_history'][0]
print('fresh pairs:', len(e['fresh_pairs']), 'allowed:', e['allowed_pairs'])
print('selection:', e['selection'])

fresh pairs: 6 allowed: 15
selection: {'pair': [10, 80], 'sign': 1, 'physical_correlation': 0.875, 'score': 0.875, 'score_rule': 'max_abs_correlation', 'score_direction': 'max', 'weight': 1.625, 'weight_power': 1, 'observed_epoch': 0, 'age': 0, 'stale': False, 'task_ids': ['r0:p0:b1'], 'contexts': [{'task_id': 'r0:p0:b1', 'nodes': [-7, 10, 80], 'parent_version': 0}]}


## Частичное обновление вокруг retained center

In [3]:
center = solver.result['selection_history'][0]['pair'][1]
cover = survivor_neighbor_cover(graph, center, max_size=3, growth_metric='count', version=1)
for task in cover: print(task['task_id'], task['nodes'])

r1:cover:80:0:0 (-7, 10, 80)
r1:cover:80:0:1 (30, 80, 101)
r1:cover:80:0:2 (-7, 80, 205)


## Полный run с bank keep_unobserved

In [4]:
config = json.loads((ROOT/'configs/solvers/incremental_ls_keep.json').read_text())
config['output'] = {'report': 0}
solver = IncrementalLSRQAOACPU(config); solution = solver.solve(graph, num_treads=1)
print('cut', solver.cut_value, 'reductions', solver.result['reductions_completed'])
print('tasks', solver.result['resources']['local_tasks_total'], 'shots', solver.result['resources']['shots_used'])
print('selections', [e['selection']['pair'] for e in solver.result['epoch_history']])

cut 18.0 reductions 2
tasks 4 shots 224
selections [[10, 80], [30, 80]]


## Стоимость и ограничения

In [5]:
print(solver.result['resources'])
print('bank entries:', len(solver.result['final_bank']))
print('checkpoint:', solver.supports_checkpoint)

{'shots_used': 224, 'training_shots': 160, 'correlation_shots': 64, 'objective_evaluations': 20, 'optimizer_steps': 4, 'readout_shots': 0, 'source_calls': 4, 'local_tasks_total': 4, 'local_tasks_completed': 4, 'reductions_completed': 2, 'epochs': 2, 'num_treads_requested': 1, 'num_treads_applied': 1, 'num_treads': 1, 'unknown_worker_spend': False, 'shots_upper_bound': 224, 'budget': {'total': 224, 'used': 224, 'remaining': 0}, 'classical_seconds': 0.022896100068464875, 'stale_selection_share': 0.0}
bank entries: 4
checkpoint: False


## Пять score: значения различаются

Score не является физическим c и не является вероятностью.

In [6]:
# Fixture B: scores are intentionally different for four rules.
# See tests/test_incremental_ls.py for the exact winner assertions.